In [1]:
# ---------- Cellule 0 : configuration et chargement de FHV renommé ----------
import os, time, shutil, warnings
import numpy as np
import pandas as pd
from IPython.display import display
warnings.filterwarnings("ignore")

RENOM_DIR = os.path.join(os.getcwd(), "renamed_data")
CLEAN_DIR = os.path.join(os.getcwd(), "clean_data")
os.makedirs(CLEAN_DIR, exist_ok=True)

TYPES = {"zone_depart": "float32", "zone_arrivee": "float32"}
DATES = ["date_debut", "date_fin"]

t0 = time.time()
fhv = pd.read_csv(os.path.join(RENOM_DIR, "fhv_renomme.csv"), dtype=TYPES)
for c in DATES:
    fhv[c] = pd.to_datetime(fhv[c], format="ISO8601", errors="coerce")
N_DEPART = len(fhv)
print(f"FHV chargé : {N_DEPART:,} lignes, {fhv.shape[1]} colonnes – "
      f"{fhv.memory_usage(deep=True).sum()/1e9:.2f} Go – {time.time()-t0:,.0f} s")
display(fhv.head())

FHV chargé : 6,407,159 lignes, 4 colonnes – 0.15 Go – 33 s


,date_debut,date_fin,zone_depart,zone_arrivee
0,2024-02-01 00:00:00,2024-02-01 00:17:00,NaN,NaN
1,2024-02-01 00:30:00,2024-02-01 01:03:00,NaN,NaN
2,2024-02-01 00:00:00,2024-02-01 00:13:00,NaN,NaN
3,2024-02-01 00:00:00,2024-02-01 00:41:00,NaN,NaN
4,2024-02-01 00:46:26,2024-02-01 01:25:00,NaN,NaN


Elle affiche info, le résumé par colonne, les doublons, le describe des zones et la période.
Effet sur les données : aucun, c’est le diagnostic de départ.

In [3]:
# ---------- Cellule 1 : description générale de FHV ----------
print("info")
fhv.info(memory_usage="deep", show_counts=True)

nb_lignes = len(fhv)
nb_doublons = int(fhv.duplicated().sum())

resume = pd.DataFrame({
    "type":                  fhv.dtypes.astype(str),
    "nb_non_null":           fhv.notna().sum(),
    "nb_null":               fhv.isna().sum(),
    "pct_null":              (fhv.isna().mean() * 100).round(3),
    "nb_valeurs_distinctes": fhv.nunique(),
    "exemple":               fhv.iloc[0],
})
print(f"\n Résumé par colonne ({nb_lignes:,} lignes, {fhv.shape[1]} colonnes)")
display(resume)

print(f"\nLignes en double (les 4 colonnes identiques) : {nb_doublons:,} ({nb_doublons/nb_lignes:.3%})")
print(f"Lignes avec au moins une valeur manquante  : {int(fhv.isna().any(axis=1).sum()):,}")

print("\ndescribe() des zones ")
display(fhv.describe(include="number", percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T.round(2))

print("\nPériode couverte ")
print(f"date_debut : du {fhv['date_debut'].min()} au {fhv['date_debut'].max()}")
print(f"date_fin   : du {fhv['date_fin'].min()} au {fhv['date_fin'].max()}")

info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6407159 entries, 0 to 6407158
Data columns (total 4 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   date_debut    6407159 non-null  datetime64[ns]
 1   date_fin      6407159 non-null  datetime64[ns]
 2   zone_depart   1055332 non-null  float32       
 3   zone_arrivee  5389013 non-null  float32       
dtypes: datetime64[ns](2), float32(2)
memory usage: 146.6 MB

 Résumé par colonne (6,407,159 lignes, 4 colonnes)


,type,nb_non_null,nb_null,pct_null,nb_valeurs_distinctes,exemple
date_debut,datetime64[ns],6407159,0,0.000,3884839,2024-02-01 00:00:00
date_fin,datetime64[ns],6407159,0,0.000,3904070,2024-02-01 00:17:00
zone_depart,float32,1055332,5351827,83.529,262,NaN
zone_arrivee,float32,5389013,1018146,15.891,263,NaN



Lignes en double (les 4 colonnes identiques) : 404,381 (6.311%)
Lignes avec au moins une valeur manquante  : 5,376,129

describe() des zones 


,count,mean,std,min,1%,25%,50%,75%,99%,max
zone_depart,1055332.0,135.39,74.48,1.0,6.0,74.0,136.0,206.0,263.0,265.0
zone_arrivee,5389013.0,136.47,79.84,1.0,5.0,69.0,135.0,213.0,265.0,265.0



Période couverte 
date_debut : du 2024-02-01 00:00:00 au 2025-03-31 23:59:47
date_fin   : du 1900-01-01 05:32:00 au 2026-03-06 02:29:00


In [4]:
# ---------- Cellule 2 : suppression des doublons ----------
avant = len(fhv)
masque_doublons = fhv.duplicated(keep="first")
nb_doublons = int(masque_doublons.sum())

print(f"Doublons exacts (les 4 colonnes identiques) : {nb_doublons:,} ({nb_doublons/avant:.3%})")
if nb_doublons:
    exemple = fhv[fhv.duplicated(keep=False)].sort_values(["date_debut", "zone_depart"]).head(6)
    print("Exemple :")
    display(exemple)

fhv = fhv[~masque_doublons].reset_index(drop=True)
print(f"Avant : {avant:,}  {len(fhv):,} lignes ({avant - len(fhv):,} supprimées)")

Doublons exacts (les 4 colonnes identiques) : 404,381 (6.311%)
Exemple :


,date_debut,date_fin,zone_depart,zone_arrivee
251,2024-02-01 00:00:09,2024-02-01 00:04:51,NaN,212.0
450,2024-02-01 00:00:09,2024-02-01 00:04:51,NaN,212.0
152,2024-02-01 00:00:11,2024-02-01 00:28:55,NaN,82.0
468,2024-02-01 00:00:11,2024-02-01 00:28:55,NaN,82.0
257,2024-02-01 00:05:28,2024-02-01 00:39:20,NaN,18.0
456,2024-02-01 00:05:28,2024-02-01 00:39:20,NaN,18.0


Avant : 6,407,159  6,002,778 lignes (404,381 supprimées)


Elle supprime les courses répétées, avec les 4 colonnes identiques.
Effet sur les données : quelques lignes en moins.

In [5]:
# ---------- Cellule 3 : valeurs vides ----------
print("Vides par colonne :")
display(pd.DataFrame({"nb_null": fhv.isna().sum(), "pct_null": (fhv.isna().mean() * 100).round(2)}))

mois = fhv["date_debut"].dt.strftime("%Y-%m")
print("Part de zones de départ vides par mois :")
display((fhv["zone_depart"].isna().groupby(mois).mean() * 100).round(2).to_frame("pct_zone_depart_vide"))

avant = len(fhv)
fhv = fhv.dropna(subset=["date_debut", "date_fin"]).reset_index(drop=True)
print(f"Courses sans date supprimées : {avant - len(fhv):,}")


Vides par colonne :


,nb_null,pct_null
date_debut,0,0.00
date_fin,0,0.00
zone_depart,4948813,82.44
zone_arrivee,1003219,16.71


Part de zones de départ vides par mois :


,pct_zone_depart_vide
date_debut,
2024-02,80.95
2024-03,80.25
2025-02,79.12
2025-03,87.15


Courses sans date supprimées : 0


Les zones vides sont traitées à la cellule 6 (départ vide -> course supprimée, arrivée vide -> course gardée)

In [6]:
# ---------- Cellule 4 : filtre sur la période (fév-mars 2024 et fév-mars 2025) ----------
MOIS_ETUDE = ["2024-02", "2024-03", "2025-02", "2025-03"]

avant = len(fhv)
mois = fhv["date_debut"].dt.strftime("%Y-%m")
dans_periode = mois.isin(MOIS_ETUDE)

print("Courses hors période")
display(mois[~dans_periode].value_counts().sort_index().to_frame("nb_courses"))

fhv = fhv[dans_periode].reset_index(drop=True)

print("Courses gardées")
display(fhv["date_debut"].dt.strftime("%Y-%m").value_counts().sort_index().to_frame("nb_courses"))

Courses hors période


,nb_courses
date_debut,


Courses gardées


,nb_courses
date_debut,
2024-02,1113799
2024-03,1405816
2025-02,1449973
2025-03,2033190


In [7]:
# ---------- Cellule 5 : calcul et contrôle de la durée ----------
DUREE_MIN, DUREE_MAX = 1, 180          # minutes

fhv["duree_min"] = ((fhv["date_fin"] - fhv["date_debut"]).dt.total_seconds() / 60).astype("float32")

print("Distribution de la durée (minutes) avant contrôle :")
display(fhv["duree_min"].describe(percentiles=[0.001, 0.01, 0.5, 0.99, 0.999]).to_frame("duree_min").T.round(2))

d = fhv["duree_min"]
print(f"Durée négative         : {int((d < 0).sum()):,}")
print(f"Durée entre 0 et 1 min : {int(((d >= 0) & (d < DUREE_MIN)).sum()):,}")
print(f"Durée > {DUREE_MAX} min        : {int((d > DUREE_MAX).sum()):,}")

invalide = (d < DUREE_MIN) | (d > DUREE_MAX)
fhv.loc[invalide, "duree_min"] = np.nan


Distribution de la durée (minutes) avant contrôle :


,count,mean,std,min,0.1%,1%,50%,99%,99.9%,max
duree_min,6002778.0,-46.03,71066.23,-65822364.0,0.05,1.55,17.8,111.57,512.0,1052988.0


Durée négative         : 246
Durée entre 0 et 1 min : 33,235
Durée > 180 min        : 26,241


In [8]:
# ---------- Cellule 6 : contrôle des zones (1 à 263) ----------
ZONE_MIN, ZONE_MAX = 1, 263

avant = len(fhv)
dep_ok = fhv["zone_depart"].between(ZONE_MIN, ZONE_MAX)       # une zone vide donne False
arr_ok = fhv["zone_arrivee"].between(ZONE_MIN, ZONE_MAX)

display(fhv.loc[~dep_ok, "zone_depart"].value_counts(dropna=False).to_frame("nb_courses"))
display(fhv.loc[~arr_ok, "zone_arrivee"].value_counts(dropna=False).to_frame("nb_courses"))

fhv.loc[~arr_ok, "zone_arrivee"] = np.nan
fhv = fhv[dep_ok].reset_index(drop=True)
fhv["zone_depart"] = fhv["zone_depart"].astype("int16")
fhv["zone_arrivee"] = fhv["zone_arrivee"].astype("Int16")    # entier qui accepte les vides

print(f"Zones de départ distinctes : {fhv['zone_depart'].nunique()}")

,nb_courses
zone_depart,
NaN,4948813
265.0,8963
264.0,18


,nb_courses
zone_arrivee,
NaN,1003219
265.0,124740
264.0,36


Zones de départ distinctes : 260


In [9]:
# ---------- Cellule 7 : bilan du nettoyage de FHV ----------
n_fin = len(fhv)
print(f"Lignes au départ  : {N_DEPART:,}")
print(f"Lignes gardées    : {n_fin:,} ({n_fin/N_DEPART:.2%})")
print(f"Lignes supprimées : {N_DEPART - n_fin:,} ({1 - n_fin/N_DEPART:.2%}) – doublons, sans date, hors période, zone de départ vide ou inconnue")

bilan = pd.DataFrame({
    "type":      fhv.dtypes.astype(str),
    "nb_vides":  fhv.isna().sum(),
    "pct_vides": (fhv.isna().mean() * 100).round(3),
})
print("\nValeurs vides par colonne :")
display(bilan)

print("Statistiques finales :")
display(fhv.describe(include="number", percentiles=[0.01, 0.5, 0.99]).T.round(2))

print(f"Période : du {fhv['date_debut'].min()} au {fhv['date_debut'].max()}")
print(f"Zones de départ distinctes : {fhv['zone_depart'].nunique()}")

Lignes au départ  : 6,407,159
Lignes gardées    : 1,044,984 (16.31%)
Lignes supprimées : 5,362,175 (83.69%) – doublons, sans date, hors période, zone de départ vide ou inconnue

Valeurs vides par colonne :


,type,nb_vides,pct_vides
date_debut,datetime64[ns],0,0.000
date_fin,datetime64[ns],0,0.000
zone_depart,int16,0,0.000
zone_arrivee,Int16,51668,4.944
duree_min,float32,17018,1.629


Statistiques finales :


,count,mean,std,min,1%,50%,99%,max
zone_depart,1044984.0,134.262559,73.876835,1.0,6.0,134.0,259.0,263.0
zone_arrivee,993316.0,133.540788,74.532208,1.0,5.0,134.0,259.0,263.0
duree_min,1027966.0,28.345463,21.207443,1.0,3.0,22.733334,102.800003,180.0


Période : du 2024-02-01 00:01:27 au 2025-03-31 23:57:12
Zones de départ distinctes : 260


In [10]:
# ---------- Cellule 8 : sauvegarde de FHV nettoyé (CSV compressé) ----------
libre = shutil.disk_usage(CLEAN_DIR).free / 1e9
print(f"Espace libre sur le disque : {libre:.2f} Go")

t0 = time.time()
cible = os.path.join(CLEAN_DIR, "fhv_clean.csv.gz")
fhv.to_csv(cible, index=False, chunksize=1_000_000, date_format="%Y-%m-%d %H:%M:%S",
           compression={"method": "gzip", "compresslevel": 1})

print(f"FHV nettoyé sauvegardé : {cible}")
print(f"{len(fhv):,} lignes, {fhv.shape[1]} colonnes – {os.path.getsize(cible)/1e9:.2f} Go – {time.time()-t0:,.0f} s")
print("Colonnes :", list(fhv.columns))

Espace libre sur le disque : 2.67 Go
FHV nettoyé sauvegardé : c:\Users\DELL\Desktop\esprit\ML\notebook_v0\Nootebook_vf\clean_data\fhv_clean.csv.gz
1,044,984 lignes, 5 colonnes – 0.02 Go – 9 s
Colonnes : ['date_debut', 'date_fin', 'zone_depart', 'zone_arrivee', 'duree_min']
